In [2]:
import glob
import pandas as pd
chunk_files = glob.glob("parquet_chunks/*.parquet")

df = pd.read_parquet("/home/tts26/sonh/Orbit-Wars-RL/gnn_rl/parquet_chunks/chunk_00000.parquet")
df.head()

,step,obs_step,player_id,action_source,action_angle,action_ships,raw_obs
0,2,1,0,4.0,-0.985460,13.0,"{'angular_velocity': 0.028384471849050706, 'co..."
1,2,1,2,6.0,-2.556257,13.0,"{'angular_velocity': 0.028384471849050706, 'co..."
2,2,1,3,7.0,2.156132,13.0,"{'angular_velocity': 0.028384471849050706, 'co..."
3,7,6,1,5.0,-0.136370,28.0,"{'angular_velocity': 0.028384471849050706, 'co..."
4,9,8,2,6.0,2.919956,21.0,"{'angular_velocity': 0.028384471849050706, 'co..."


In [9]:
print(df.dtypes)
print(f"Data Shape: {df.shape}")

step               int64
obs_step           int64
player_id          int64
action_source    float64
action_angle     float64
action_ships     float64
raw_obs           object
dtype: object
Data Shape: (371283, 7)


In [ ]:
sample = df["raw_obs"].iloc[0]

print(type(sample))
print(sample)

<class 'dict'>
{'angular_velocity': 0.028384471849050706, 'comet_planet_ids': array([], dtype=int64), 'comets': array([], dtype=object), 'fleets': array([], dtype=object), 'initial_planets': array([array([ 0.        , -1.        , 96.92510434, 62.6384318 ,  2.60943791,
              11.        ,  5.        ])                                      ,
       array([ 1.        , -1.        , 37.3615682 , 96.92510434,  2.60943791,
              11.        ,  5.        ])                                      ,
       array([ 2.        , -1.        , 62.6384318 ,  3.07489566,  2.60943791,
              11.        ,  5.        ])                                      ,
       array([ 3.        , -1.        ,  3.07489566, 37.3615682 ,  2.60943791,
              11.        ,  5.        ])                                      ,
       array([ 4.        , -1.        , 78.25172464, 90.81098899,  2.09861229,
              26.        ,  3.        ])                                      ,
       array([

In [7]:
from datasets import replay_to_dataframe, _normalise_paths
import pandas as pd
df_json = replay_to_dataframe("/home/tts26/sonh/Orbit-Wars-RL/gnn_rl/replays/77249937.json")

print(f"Data Shape: {df_json.shape}")
df_json.dtypes

/home/tts26/miniconda3/envs/orbit_wars/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data Shape: (315, 7)


step               int64
obs_step           int64
player_id          int64
action_source    float64
action_angle     float64
action_ships     float64
raw_obs           object
dtype: object

In [8]:
sample = df_json["raw_obs"].iloc[0]

print(type(sample))
print(sample)

<class 'dict'>
{'angular_velocity': 0.028384471849050706, 'comet_planet_ids': [], 'comets': [], 'fleets': [], 'initial_planets': [[0, -1, 96.92510434311379, 62.63843180081058, 2.6094379124341005, 11, 5], [1, -1, 37.36156819918942, 96.92510434311379, 2.6094379124341005, 11, 5], [2, -1, 62.63843180081058, 3.0748956568862127, 2.6094379124341005, 11, 5], [3, -1, 3.0748956568862127, 37.36156819918942, 2.6094379124341005, 11, 5], [4, -1, 78.251724635019, 90.81098898515404, 2.09861228866811, 26, 3], [5, -1, 9.18901101484596, 78.251724635019, 2.09861228866811, 26, 3], [6, -1, 90.81098898515404, 21.748275364980998, 2.09861228866811, 26, 3], [7, -1, 21.748275364980998, 9.18901101484596, 2.09861228866811, 26, 3], [8, -1, 93.03771089891809, 93.00732023145017, 2.6094379124341005, 87, 5], [9, -1, 6.992679768549834, 93.03771089891809, 2.6094379124341005, 87, 5], [10, -1, 93.00732023145017, 6.9622891010819075, 2.6094379124341005, 87, 5], [11, -1, 6.9622891010819075, 6.992679768549834, 2.60943791243410

In [1]:
%load_ext autoreload
%autoreload 2
import torch
from kaggle_environments import make
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from kaggle_agent_wrapper import KaggleGNNWrapper
from minhtuuse import AggressiveNearestAgent

# 1. Initialize the environment
env = make("orbit_wars", configuration={"seed": 2}, debug=True)

# 2. Load the trained Imitation Learning agent
trained_il_agent = KaggleGNNWrapper(model_path="/home/tts26/sonh/Orbit-Wars-RL/gnn_rl/artifacts/gnn_il.pt", device="cuda" if torch.cuda.is_available() else "cpu")

# 3. Load baseline opponents
opponent = AggressiveNearestAgent()

# 4. Run the match
# Adjust the list order based on how many players the environment supports
print("Starting evaluation match...")
env.run([trained_il_agent.act, opponent.act])

# 5. Print results
final = env.steps[-1]
print("\n--- Match Results ---")
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

# 6. Render
env.render(mode="ipython", width=800, height=600)

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 23.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_go
[kaggle_environments.envs.open_s

/home/tts26/miniconda3/envs/orbit_wars/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: Error(s) in loading state_dict for GNNAgent:
	Missing key(s) in state_dict: "backbone.dummy_embedding", "backbone.convs.0.att", "backbone.convs.0.bias", "backbone.convs.0.lin_l.weight", "backbone.convs.0.lin_l.bias", "backbone.convs.0.lin_r.weight", "backbone.convs.0.lin_r.bias", "backbone.convs.0.lin_edge.weight", "backbone.convs.1.att", "backbone.convs.1.bias", "backbone.convs.1.lin_l.weight", "backbone.convs.1.lin_l.bias", "backbone.convs.1.lin_r.weight", "backbone.convs.1.lin_r.bias", "backbone.convs.1.lin_edge.weight", "backbone.convs.2.att", "backbone.convs.2.bias", "backbone.convs.2.lin_l.weight", "backbone.convs.2.lin_l.bias", "backbone.convs.2.lin_r.weight", "backbone.convs.2.lin_r.bias", "backbone.convs.2.lin_edge.weight", "lstm.weight_ih", "lstm.weight_hh", "lstm.bias_ih", "lstm.bias_hh". 
	Unexpected key(s) in state_dict: "backbone.convs.0.eps", "backbone.convs.0.nn.0.weight", "backbone.convs.0.nn.0.bias", "backbone.convs.0.nn.3.weight", "backbone.convs.0.nn.3